## Importer av paket och data från REMbox

In [ ]:
import pandas as pd 
import hvplot.pandas #noqa #plotpaket
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from rembox_integration_tools import REMboxDataQuery
from rembox_integration_tools.rembox_analysis import StudyColumn, SeriesColumn
from pathlib import Path

# om plotly önskas så skrivs följande hvplot.extension("plotly")
hvplot.extension("bokeh")

CLIENT_ID_ENV_VAR = "REMBOX_INT_CLIENT_ID"
CLIENT_PWD_ENV_VAR = "REMBOX_INT_CLIENT_PWD"
TOKEN_URI = "https://autoqa.vll.se/dpqaauth/connect/token" #Var finns access token
API_URI = "https://rembox.vll.se/api" #Var finns API:t
ORIGIN_URI = "https://rembox.vll.se" #Vilken URL

rembox = REMboxDataQuery(
    client_id_environment_variable=CLIENT_ID_ENV_VAR,
    client_secret_environment_variable=CLIENT_PWD_ENV_VAR,
    token_uri=TOKEN_URI,
    api_uri=API_URI,
    origin_uri=ORIGIN_URI
)

valid_study_columns = StudyColumn()
valid_series_columns = SeriesColumn()

In [ ]:
rembox.reset_filter_options()
def get_data_from_fluoro(rembox: REMboxDataQuery) -> tuple[pd.DataFrame, pd.DataFrame]:

    
    rembox.filter_options.set_inclusive_tags(
        machine_types=["XASTAT"],     # CT-CT, Fluoroscopic-XASTAT, Mobile C-arm-XAMOB, Conventional-DX, Mammography-MG, Intraoral-IO, Panoramic-PX, Dental Cone Beam CT-DCBCT, PET-PET, PET/CT-PETCT, SPECT-SPECT, SPECT/CT-SPECTCT, Nuclear Medicine-NM, Mobile X-ray-DXMOB, Conventional with fluoro-DXXA
        machines=["U105", "U106", "U104"]
    )
    
    #rembox.filter_options.set_exclusive_tags() om jag vill ange filter där man bortser från ett visst kriterie

    rembox.filter_options.patient_age_interval_include_nulls = True
    
    rembox.filter_options.study_time_interval_start_date = "2025-01-01T00:00:00Z"
    rembox.filter_options.study_time_interval_end_date = "2026-05-31T00:00:00Z" #Lägg till en dag till önskad tidsperiod för att matcha GUI
    
    rembox.deanonymize_performing_physician = False
    
    rembox.add_columns(
        columns=[
            valid_study_columns.Id,
            valid_study_columns.StudyDateTime,
            valid_study_columns.AccessionNumber,
            valid_study_columns.PatientDbId,
            valid_study_columns.PatientId,
            valid_study_columns.PatientsSex,
            valid_study_columns.StudyDescription,
            valid_study_columns.StudyId,
            valid_study_columns.StudyInstanceUID,
            valid_study_columns.ProcedureCode,
            valid_study_columns.ProcedureCodeMeaning,
            valid_study_columns.ProtocolCode,
            valid_study_columns.ProtocolCodeMeaning,
            valid_study_columns.DoseAreaProductTotal,
            valid_study_columns.AcquisitionDoseAreaProductTotal,
            valid_study_columns.FluoroDoseAreaProductTotal,
            valid_study_columns.DoseRPTotal,
            valid_study_columns.AcquisitionDoseRPTotal,
            valid_study_columns.FluoroDoseRPTotal,
            valid_study_columns.TotalAcquisitionTime,
            valid_study_columns.TotalFluoroTime,
            valid_study_columns.TotalNumberOfIrradiationEvents,
            valid_study_columns.TotalNumberOfRadiographicFrames,
            valid_study_columns.AcquisitionPlane,
            valid_study_columns.Machine,
            valid_study_columns.PerformingPhysicianName,
            valid_series_columns.DateTimeStarted,
            valid_series_columns.AcquisitionPlaneSeries,
            valid_series_columns.AcquisitionProtocol,
            valid_series_columns.AcquisitionType,
            valid_series_columns.FluoroMode,
            valid_series_columns.IrradiationEventType,
            valid_series_columns.IrradiationEventUID,
            valid_series_columns.DoseAreaProduct,
            valid_series_columns.DoseRP,
            valid_series_columns.EntranceExposureAtRP,
            valid_series_columns.NumberOfPulses,
            valid_series_columns.PulseRate,
            valid_series_columns.PatientEquivalentThickness,
            valid_series_columns.PositionerPrimaryAngle,
            valid_series_columns.PositionerSecondaryAngle,
            valid_series_columns.XrayFilterAluminumEquivalent,
            valid_series_columns.XrayFilterMaterial,
            valid_series_columns.XrayFilterThicknessMaximum,
            valid_series_columns.XrayFilterType
        ]
    )

    return rembox.run_query()

In [ ]:
#Hämta data från REMbox
study_data, series_data = get_data_from_fluoro(rembox=rembox)

## Kontroller av data och hantering av dataframes

In [ ]:
study = study_data.copy() #skapa kopia av dataframe på study-nivå för att kunna behålla orginalet
series = series_data.copy() #skapa kopia av dataframe på serie-nivå för att kunna behålla orginalet

In [ ]:
antal_pedaltramp = series["irradiationEventUID"].count()
patient_list = series["irradiationEventUID"].nunique()

series.drop_duplicates(subset="irradiationEventUID", keep="first", inplace=True)

antal_unika_pedaltramp = series["irradiationEventUID"].count()
unika_patienter_list = series["irradiationEventUID"].nunique()

antal_studier = study["studyInstanceUID"].count()

print(antal_pedaltramp)
print(patient_list)
print('--------')
print(antal_unika_pedaltramp)
print(unika_patienter_list)
print(antal_studier)

In [ ]:
series.head()

## Lägg till operatörsnamn

In [ ]:
#Översättningstabell från pseudo-operatörer till operatörer
names_data_path = "C:/Projekt/GIT/rvbrtg/Data/input_data/operators_2026.xlsx"
names = pd.read_excel(names_data_path)
names.columns = ["performingPhysicianName", "OperatorName"]
study_names = study.merge(names, on = ["performingPhysicianName"], how = "left")

#study_names.head()

In [ ]:
test = study_names.OperatorName.unique()

test

In [ ]:
#Operatörer som saknas? Export till csv för att komplettera.

study_names.to_csv("C:/Projekt/GIT/rvbrtg/Data/output_data/Operators_missing_IR.csv")

In [ ]:
# Fyll på med operatörer som saknas
missing_names_data_path = "C:/Projekt/GIT/rvbrtg/Data/input_data/IR_utan_operatör_2025.xlsx" #TODO Skapa denna
missing_names = pd.read_excel(missing_names_data_path, header=None)
missing_names.columns = ["accessionNumber", "OperatorName"]
study_names = study_names.merge(missing_names, on = ["accessionNumber"], how = "left")

#print(study_names[study_names.accessionNumber == 'SETUME0007821975'].OperatorName)
#print(missing_names[missing_names.accessionNumber == 'SETUME0007821975'].OperatorName)

study_names.OperatorName_x.fillna(study_names.OperatorName_y, inplace=True)
study_names = study_names.drop('OperatorName_y', axis=1)
study_names = study_names.rename(columns={'OperatorName_x': 'OperatorName'})

#study_names[study_names.accessionNumber == 'SETUME0007821975'].head()

In [ ]:
#operatörer på PCI

study_operator = study_names[study_names.OperatorName == 'AnnCatrine Strandén']

In [ ]:
print(study_operator.studyInstanceUID.count())
operator = study_operator.groupby(["studyDescription"])["doseAreaProductTotal"].sum().reset_index()
operator

In [ ]:
#Median DAP per ingreppstyp och operatör
median_KAP_studytype = study_names.groupby(
    ["OperatorName", "studyDescription"])["doseAreaProductTotal"].median().reset_index()
median_KAP_studytype

In [ ]:
#TODO Gör om denna att visa en operatörs ingrepp
fig = px.bar(median_KAP_studytype[median_KAP_studytype.studyDescription == "Coronarangiografi"], x="OperatorName", y="doseAreaProductTotal")
fig.show()

## Läs in och fusionera persondosimetri

In [ ]:
data_path_landauer = 'C:/Projekt/GIT/rvbrtg/Data/input_data/19248_allt_2025_2026.csv' #TODO Skapa denna

data_all_operators = pd.read_csv(data_path_landauer)
#data_all_operators.columns =["name","measurement_period_center"]

data_all_operators

In [ ]:
body = data_all_operators[data_all_operators.dosimeter_placement == 'Helkropp'] #TODO Ändra till öga

operator_landauer = body[body['name'] == 'STRANDEN, ANNCATRINE'] #TODO kolla denna

time_selection = operator_landauer[operator_landauer['measurement_period_center'] >= '2025-01-01']

def get_rembox_sum(row) -> float:
    return study_operator.doseAreaProductTotal[
        (study_operator['studyDateTime'] >= row.measurement_period_start) & (study_operator['studyDateTime'] <= row.measurement_period_stop)
    ].sum()

time_selection["dos_summa"] = [
    get_rembox_sum(row)
    for row in time_selection.itertuples()
]

#time_selection

In [ ]:

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add traces
fig.add_trace(
    go.Scatter(x=operator_landauer.measurement_period_center,
               y=operator_landauer.hp10,
               mode='markers',
               name='persondosimetri (Hp10)'),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=operator_landauer.measurement_period_center,
               y=operator_landauer.dos_summa,
               mode='markers',
               name='DAP REMbox'),
    secondary_y=True,
)

# Add figure title
fig.update_layout(
    title_text="Hp10 och total DAP per månad"
)

# Set x-axis title
fig.update_xaxes(
    title_text="Date",
    dtick="M1",
    tickformat="%b\n%Y")

# Set y-axes titles
fig.update_yaxes(title_text="Hp3", secondary_y=False)
fig.update_yaxes(title_text="Average DAP", secondary_y=True)

fig.show()


In [ ]:
# Create figure with secondary y-axis
fig = make_subplots(rows=3, cols=1, specs=[[{"secondary_y":True}], [{"secondary_y":True}], [{"secondary_y":True}]], shared_xaxes=True)#specs=[[{"secondary_y": True}]])

# Add traces
fig.add_trace(
    go.Scatter(x=urval_jonas_pci.measurement_period_center,
               y=urval_jonas_pci.hp10,
               mode='markers',
               marker_color='red',
               name='Jonas (Hp10)'),
    secondary_y=False,
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=urval_jonas_pci.measurement_period_center,
               y=urval_jonas_pci.dos_summa,
               mode='markers',
               marker_color='blue',
               name='Jonas (DAP)'),
    secondary_y=True,
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=urval_bjorn_pci.measurement_period_center,
               y=urval_bjorn_pci.hp10,
               mode='markers',
               marker_color='red',
               name='Björn (Hp10)'),
    secondary_y=False,
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=urval_bjorn_pci.measurement_period_center,
               y=urval_bjorn_pci.dos_summa,
               mode='markers',
               marker_color='blue',
               name='Björn (DAP)'),
    secondary_y=True,
    row=2, col=1
)


fig.add_trace(
    go.Scatter(x=urval_henrik_pci.measurement_period_center,
               y=urval_henrik_pci.hp10,
               mode='markers',
               marker_color='red',
               name='Henrik (Hp10)'),
    secondary_y=False,
    row=3, col=1
)

fig.add_trace(
    go.Scatter(x=urval_henrik_pci.measurement_period_center,
               y=urval_henrik_pci.dos_summa,
               mode='markers',
               marker_color='blue',
               name='Henrik (DAP)'),
    secondary_y=True,
    row=3, col=1
)

# Add figure title
fig.update_layout(
    height=800, width=1300,
    title_text="Hp10 and total DAP per month"
)

# Set x-axis title
fig.update_xaxes(
    #title_text="Date",
    dtick="M1",
    tickformat="%b\n%Y")

# Set y-axes titles
fig.update_yaxes(title_text="Hp10",range=[0,1.1], secondary_y=False)
fig.update_yaxes(title_text="Average DAP",range=[0,1100], secondary_y=True)

fig.show()

## Analysera projektioner

In [ ]:
#Kontrollera vad kolumnerna heter
#series.head()
print(study.studyInstanceUID)
print("-------------------------------")
print(series.studyInstanceUID)

In [ ]:
#Joina dataframes för att få study och series i samma dataframe
study_series_names = study_names.merge(series, on=["studyInstanceUID"], how="left")

study_series_names = study_series_names.drop('accessionNumber_y', axis=1)
study_series_names = study_series_names.rename(columns={'accessionNumber_x': 'accessionNumber'})
study_series_names = study_series_names.drop('studyId_y', axis=1)
study_series_names = study_series_names.rename(columns={'studyId_x': 'studyId'})

#Print för att kolla så att det funkade
print(study_series_names.columns)

In [ ]:
series_bjorn = study_series_names[study_series_names.OperatorName == 'Björn Pettersson']
series_henrik = study_series_names[study_series_names.OperatorName == 'Henrik Hagström']
series_jonas = study_series_names[study_series_names.OperatorName == 'Jonas Andersson']
series_jacob = study_series_names[study_series_names.OperatorName == 'Jacob Hasslow']
series_vikarie = study_series_names[study_series_names.OperatorName == 'Vikarie PCI']

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Alla + vikarie", "Björn", "Henrik", "Jonas"))

# Add traces
fig.add_trace(
    go.Scatter(x=series.positionerPrimaryAngle,
               y=series.positionerSecondaryAngle,
               mode='markers',
               #marker_color='red',
               name='All'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=series_vikarie.positionerPrimaryAngle,
               y=series_vikarie.positionerSecondaryAngle,
               mode='markers',
               #marker_color='#ab63fa',
               name='Vikarie'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=series_bjorn.positionerPrimaryAngle,
               y=series_bjorn.positionerSecondaryAngle,
               mode='markers',
               #marker_color='red',
               name='Björn'),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=series_henrik.positionerPrimaryAngle,
               y=series_henrik.positionerSecondaryAngle,
               mode='markers',
               #marker_color='red',
               name='Henrik'),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=series_jonas.positionerPrimaryAngle,
               y=series_jonas.positionerSecondaryAngle,
               mode='markers',
               #marker_color='#ef553b',
               name='Jonas'),
    row=2, col=2
)



#fig = px.scatter(series, x='positionerPrimaryAngle', y='positionerSecondaryAngle')

# Add figure title
fig.update_layout(
    height=1000, width=1000,
    title_text="Projections used"
)

fig.update_traces(
    marker_size=2
)

# Add figure title
fig.update_layout(
    height=1000, width=1300,
    title_text="Scatterplot per operator"
)

# Set x-axis title
fig.update_xaxes(
    title_text="Primary Angle: RAO to LAO",
    range=[-95, 95]
)

# Set y-axes titles
fig.update_yaxes(
    title_text="Secondary Angle: CAUD to CRAN",
    range=[-60, 60]
)

fig.show()

In [ ]:
LAO_jonas_pci = series_jonas[series_jonas['positionerPrimaryAngle'] >= 80]

LAO_jonas_pci

In [ ]:
high_kvp = study_series_names[study_series_names.kVp > 110]

fig = px.scatter(high_kvp, x="positionerPrimaryAngle", y="positionerSecondaryAngle", size="doseRP", color="kVp")
fig.show()

In [ ]:
fig = px.scatter(study_series_names, x="patientEquivalentThickness", y="kVp", color="kVp")
fig.show()

In [ ]:
#normalized analysis

study_series_norm = series_vikarie.copy()

study_series_norm['beamtime'] = study_series_norm['numberOfPulses'] * study_series_norm['pulseWidth']
study_series_norm['normDoseRP'] = study_series_norm['doseRP'] / study_series_norm['beamtime']
study_series_norm['normDAP'] = study_series_norm['doseAreaProduct'] / study_series_norm['beamtime']

study_series_norm.head()

In [ ]:
#high_kvp = study_series_names[study_series_names.kVp > 110]

fig = px.scatter(study_series_norm, x="positionerPrimaryAngle", y="positionerSecondaryAngle", color="normDAP")
fig.update_yaxes(title_text="Secondary Angle: CAUD to CRAN")
fig.update_xaxes(title_text="Primary Angle: RAO to LAO")
fig.show()

In [ ]:
pat_norm = study_series_names[study_series_names.accessionNumber == 'SETUME0008228780'].copy()

pat_norm['beamtime'] = pat_norm['numberOfPulses'] * pat_norm['pulseWidth']
pat_norm['normDoseRP'] = pat_norm['doseRP'] / pat_norm['beamtime']
pat_norm['normDAP'] = pat_norm['doseAreaProduct'] / pat_norm['beamtime']

#pat_norm.head()

fig = px.scatter(pat_norm, x="positionerPrimaryAngle", y="positionerSecondaryAngle", size="normDoseRP", color='irradiationEventType')
#fig.update_layout(yaxis_range=[-65,65], xaxis_range=[-65, 65], autosize=False, width=1000, height=800)
fig.update_yaxes(title_text="Secondary Angle: CAUD to CRAN")
fig.update_xaxes(title_text="Primary Angle: RAO to LAO")

fig.show()


## Export till csv

In [ ]:
#Export av data till csv
#study_data.to_csv("C:/Users/chgr09/GIT/rvbrtg/Data/output_data/XA_study_2023.csv")
#series.to_csv("C:/Users/chgr09/GIT/rvbrtg/Data/output_data/U602_series_maj.csv")
study_jonas.to_csv("C:/Users/chgr09/GIT/rvbrtg/Data/output_data/study_jonas.csv")